# CMU 11-868: LLM Inference with JAX (TPU v5e)

Run this notebook in a **TPU v5e** environment. Implement each section inside `# BEGIN Assignment 1.x` ... `# END Assignment 1.x` in `model/`, then run the test cell below each problem and the full grading cell at the end.

| Problem | File | Functions |
| :--- | :--- | :--- |
| **1.1** | `model/layers.py` | `rms_norm`, `apply_rope`, `swiglu_mlp` |
| **1.2** | `model/kv_cache.py` | `init_kv_cache`, `update_kv_cache` |
| **1.3** | `model/attention.py` | `make_attention_mask`, `gqa_attention` |
| **1.4** | `model/transformer.py` | `transformer_block`, `prefill`, `decode_step` |
| **1.5** | `model/generate.py` | `sample_next_token`, `generate` |

In [ ]:
%load_ext autoreload
%autoreload 2

import time
import jax
import jax.numpy as jnp
import numpy as np
import model
from tests.test_inference import run_all, run_section

print(f"JAX {jax.__version__} | backend={jax.default_backend()} | devices={jax.devices()}")

### JAX Primitives Reference
The cell below demonstrates the five JAX primitives used across Problems 1.1–1.5: functional slice updates (`jax.lax.dynamic_update_slice`), explicit PRNG splitting (`jax.random.split`), parameter PyTrees, single-trace static-shape `@jax.jit`, and compiled loops (`jax.lax.scan`).

Reference: [JAX Key Concepts](https://docs.jax.dev/en/latest/key-concepts.html) · [JAX API Reference](https://docs.jax.dev/en/latest/jax.html) · [Thinking in JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html)

In [ ]:
# JAX primitives reference: dynamic_update_slice, PRNG split, PyTrees, static-shape JIT, lax.scan
buf = jnp.zeros((2, 6), dtype=jnp.float32)
buf_updated = jax.lax.dynamic_update_slice(buf, jnp.array([[10.0, 20.0], [30.0, 40.0]]), start_indices=(0, 2))
print("dynamic_update_slice:\n", buf_updated)

key = jax.random.PRNGKey(0)
key, k1, k2 = jax.random.split(key, 3)
print("PRNG split:", jax.random.normal(k1, (2,)), jax.random.normal(k2, (2,)))

cfg_demo = model.ModelConfig.from_json("configs/model_test.json")
params_demo = model.init_model_params(cfg_demo, key)
print("PyTree leaves:", len(jax.tree_util.tree_leaves(params_demo)))

traces = 0

@jax.jit
def write_one(arr, val, idx):
    global traces
    traces += 1
    return jax.lax.dynamic_update_slice(arr, val, (0, idx))

arr = jnp.zeros((2, 8))
for step in range(4):
    arr = write_one(arr, jnp.ones((2, 1)) * (step + 1), jnp.asarray(step, dtype=jnp.int32))
print(f"JIT trace count (4 dynamic indices): {traces}")

_, history = jax.lax.scan(lambda c, _: (c + 1, c + 1), init=jnp.array(0), xs=None, length=5)
print("lax.scan history:", history)

---
## Problem 1.1: Core LLaMA Layers (`model/layers.py`)

### `rms_norm(x, weight, eps=1e-6)`
$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \odot w$$
- **Contract**: `x: [..., d_model]`, `weight: [d_model]` $\rightarrow$ `[..., d_model]`
- **Constraints**: Accumulate normalization in `float32`; return the same `dtype` as `x` (including `bfloat16`).

### `apply_rope(x, positions, cos, sin)`
Using the **rotate-half** convention, split each head vector $x = [x^{(1)}, x^{(2)}]$ with $x^{(1)}, x^{(2)} \in \mathbb{R}^{D/2}$ and rotate by angle table $(\cos, \sin)$ indexed by token position $p$:
$$\text{RoPE}(x, p) = \big[x^{(1)} \odot \cos_p - x^{(2)} \odot \sin_p,\;\; x^{(1)} \odot \sin_p + x^{(2)} \odot \cos_p\big]$$
- **Contract**: `x: [B, T, H, D]`, `positions: [B, T]` (`int32`), `cos, sin: [max_seq_len, D // 2]` (`float32`) $\rightarrow$ `[B, T, H, D]`
- **Constraints**: Compute rotation in `float32`; preserve `x.dtype`, vector norms, and relative-position inner products.

### `swiglu_mlp(x, w_gate, w_up, w_down)`
$$\text{SwiGLU}(x) = \big(\text{SiLU}(x W_{\text{gate}}) \odot (x W_{\text{up}})\big) W_{\text{down}}$$
- **Contract**: `x: [B, T, d_model]`, `w_gate, w_up: [d_model, d_ff]`, `w_down: [d_ff, d_model]` $\rightarrow$ `[B, T, d_model]`

In [ ]:
run_section(model, "1.1")

---
## Problem 1.2: Static-Shape KV Cache (`model/kv_cache.py`)

![Left-Padded Static KV Cache Layout](figures/kv_cache_left_pad.svg)

*Left-padded static KV cache ($B = 2, P = 5, S = 8$): aligning the last real prompt token at index $P - 1$ lets every sequence write decode step $t$ to the same slot `cache_index = P + t` while tracking independent RoPE positions.*

### `init_kv_cache(config, batch_size, max_seq_len, dtype=jnp.float32)`
- **Contract**: Returns a `tuple` of `config.n_layers` `LayerKVCache(k, v)` entries, each zero-initialized with shape `[B, max_seq_len, n_kv_heads, head_dim]` and `dtype`.

### `update_kv_cache(layer_cache, k_new, v_new, cache_index)`
- **Contract**: `layer_cache.{k,v}: [B, S, H_kv, D]`, `k_new, v_new: [B, T, H_kv, D]`, `cache_index: scalar int32` $\rightarrow$ updated `LayerKVCache` with identical static shapes.
- **Constraints**: Writes `k_new` and `v_new` into sequence slots `[cache_index, cache_index + T)` without mutating the input cache, casts to the cache `dtype`, and must trace **once** under `@jax.jit` across varying `cache_index` values.

In [ ]:
run_section(model, "1.2")

---
## Problem 1.3: Masking & Grouped-Query Attention (`model/attention.py`)

![Grouped-Query Attention](figures/gqa_grouping.svg)

*Grouped-Query Attention ($H = 4, H_{\text{kv}} = 2, G = H / H_{\text{kv}} = 2$): query heads are grouped as `[B, T, H_kv, G, D]` so each KV head is shared without duplicating `k_cache` or `v_cache`.*

### `make_attention_mask(cache_valid_mask, cache_index, query_len)`
- **Contract**: `cache_valid_mask: [B, S]` (`bool`), `cache_index: scalar int32`, `query_len: static int T` $\rightarrow$ `[B, 1, T, S]` (`bool`, `True` = attend).
- **Semantics**: Entry `(b, 0, t, s)` is `True` iff key slot `s` is valid (`cache_valid_mask[b, s]`) **and** causal ($s \le \text{cache\_index} + t$).

### `gqa_attention(q, k_cache, v_cache, attn_mask)`
$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{D}} + M\right) V$$
- **Contract**: `q: [B, T, H, D]`, `k_cache, v_cache: [B, S, H_kv, D]`, `attn_mask: [B, 1, T, S]` $\rightarrow$ `[B, T, H, D]` (`q.dtype`).
- **Constraints**:
  - Each KV head serves `H // H_kv` query heads without copying K/V caches.
  - Compute attention scores and `softmax` in `float32`.
  - Masked-out slots must have zero effect on outputs, and all-masked query rows (from left-padding) must produce finite values (no `NaN`/`Inf`).

In [ ]:
run_section(model, "1.3")

---
## Problem 1.4: Transformer Block, Prefill, and Decode Step (`model/transformer.py`)

![Pre-Norm LLaMA Transformer Block](figures/llama_block.svg)

*Pre-norm LLaMA Transformer block: input `x` (`[B, T, d_model]`) $\rightarrow$ output (`[B, T, d_model]`) and updated `LayerKVCache`.*

### `transformer_block(x, layer_params, layer_cache, positions, cos, sin, cache_index, attn_mask, config)`
- **Contract**: `x: [B, T, d_model]`, `positions: [B, T]`, `attn_mask: [B, 1, T, S]` $\rightarrow$ `(out: [B, T, d_model], new_layer_cache)`.
- **Semantics**: Pre-norm LLaMA decoder block as illustrated above (RMSNorm $\rightarrow$ QKV + RoPE $\rightarrow$ KV-cache update $\rightarrow$ GQA + $W_o$ + residual $\rightarrow$ RMSNorm $\rightarrow$ SwiGLU + residual).

### `prefill(params, prompt_ids, prompt_mask, config, max_seq_len, cos, sin)`
- **Contract**: `prompt_ids, prompt_mask: [B, P]` $\rightarrow$ `(logits: [B, P, V], cache, cache_valid_mask: [B, S], next_positions: [B, 1])`.
- **Constraints**:
  - Must support left-padded prompts: real tokens of sequence $b$ receive RoPE positions $0, 1, \dots, L_b - 1$ (pad tokens $0$), and `next_positions[b, 0]` $= L_b$.
  - Writes prompt K/V into cache slots `[0, P)` and supports both tied (`embed_tokens.T`) and untied (`lm_head`) output projections.

### `decode_step(params, token_ids, positions, cache, cache_valid_mask, cache_index, cos, sin, config)`
- **Contract**: `token_ids: [B, 1]`, `positions: [B, 1]`, `cache_index: scalar int32` $\rightarrow$ `(logits: [B, 1, V], new_cache, new_cache_valid_mask)`.
- **Constraints**: Marks `cache_index` valid, writes the single-token K/V at `cache_index`, and must trace **once** under `@jax.jit` across consecutive decode steps.

In [ ]:
run_section(model, "1.4")

---
## Problem 1.5: Sampling & Batched Generation (`model/generate.py`)

### `sample_next_token(logits, rng_key, temperature=0.0, top_k=0)`
- **Contract**: `logits: [B, V]` $\rightarrow$ `[B]` (`int32`).
- **Semantics**: Greedy `argmax` when `temperature <= 0.0`; otherwise temperature-scaled categorical sampling, restricted to the `top_k` highest logits when `0 < top_k < V`.

### `generate(params, prompt_ids, prompt_mask, config, max_new_tokens, rng_key, temperature=0.0, top_k=0)`
- **Contract**: `prompt_ids, prompt_mask: [B, P]` $\rightarrow$ `generated_ids: [B, max_new_tokens]` (`int32`, excluding prompt).
- **Constraints**:
  - Compiled with `@jax.jit` (static: `config`, `max_new_tokens`, `temperature`, `top_k`) and uses `jax.lax.scan` for autoregressive decode (no Python decode loop).
  - Must handle `max_new_tokens == 1`, split PRNG keys per step for deterministic stochastic sampling, and produce identical tokens for left-padded batched prompts vs. unpadded `B = 1` runs.

In [ ]:
run_section(model, "1.5")

---
## Full Grading Suite & Roofline Benchmark (TPU v5e)

Runs all correctness tests, the TPU v5e KV-cache speedup test ($\ge 100\times$ over uncached recomputation on TPU v5e), and the batch-size roofline benchmark in-process.

In [ ]:
exit_code = run_all(model, benchmark=True)
assert exit_code == 0, "Some tests failed! See output above."

---
## Optional: HuggingFace LLaMA Checkpoint Generation

In [ ]:
RUN_HF_DEMO = False

if RUN_HF_DEMO:
    model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
    dtype = jnp.bfloat16 if jax.default_backend() in ("tpu", "gpu") else jnp.float32
    cfg_hf, params_hf, tok = model.load_hf_llama_weights(model_id, dtype=dtype)

    texts = [
        "The key advantage of a static-shape KV cache on TPUs is",
        "In one sentence, explain why LLM decode is memory-bandwidth bound:",
    ]
    encoded = [tok.encode(t) for t in texts]
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    p_ids, p_mask = model.left_pad_prompts(encoded, pad_token_id=pad_id)

    t0 = time.perf_counter()
    gen_ids = model.generate(params_hf, p_ids, p_mask, cfg_hf, max_new_tokens=48, rng_key=jax.random.PRNGKey(0), temperature=0.0)
    gen_ids.block_until_ready()
    print(f"Compile + first run: {(time.perf_counter() - t0)*1e3:.1f} ms")

    t0 = time.perf_counter()
    gen_ids = model.generate(params_hf, p_ids, p_mask, cfg_hf, max_new_tokens=48, rng_key=jax.random.PRNGKey(0), temperature=0.0)
    gen_ids.block_until_ready()
    dt = time.perf_counter() - t0
    print(f"Steady-state: {dt*1e3:.1f} ms ({(len(texts)*48)/dt:.1f} tok/s)\n")

    for prompt_text, out_tokens in zip(texts, np.asarray(gen_ids)):
        print("PROMPT:", prompt_text)
        print("OUTPUT:", tok.decode(out_tokens, skip_special_tokens=True))
        print("-" * 60)
else:
    print("Set RUN_HF_DEMO = True to run generation with SmolLM2-135M.")